In [1]:
exec(open('/choreonoid_ws/install/share/irsl_choreonoid/sample/irsl_import.py').read())

In [2]:
import os
os.environ['ROS_MASTER_URI'] = 'http://133.15.97.117:11311'
#os.environ['ROS_MASTER_URI'] = 'http://133.15.97.61:11400'
os.environ['ROS_IP'] = '133.15.97.61'
os.environ['ROS_HOSTNAME'] = '133.15.97.61'

In [3]:
import rospy
from IPython.display import display
import ipywidgets as widgets
from threading import Lock
from sensor_msgs.msg import Joy

In [4]:
ri = RobotInterface('robot_interface.yaml')

loading model from //userdir/SimpleHand/simplehand.body
joint: [{'name': 'default', 'topic': '/divided_robot/trajectory_controller/command', 'type': 'command', 'joint_names': ['LINK_0', 'LINK_1', 'LINK_2', 'LINK_3', 'LINK_4', 'LINK_5']}, {'name': 'gripper', 'topic': '/divided_robot/gripper_controller/command', 'type': 'command', 'joint_names': ['LINK_6']}]
devices: [{'topic': '/divided_robot/joint_states', 'class': 'JointState', 'name': 'joint_state'}, {'topic': '/divided_robot/trajectory_controller/state', 'class': 'JointTrajectoryState', 'name': 'joint_trajectory_state'}, {'topic': '/divided_robot/Camera0/value', 'type': 'std_msgs/Float64', 'name': 'Camera0', 'rate': 10}]


In [5]:
EE_LINK = "LINK_5"
HAND_LINK = "LINK_6"
resetAngle = [-0.00,  1.28,  0.77466035, 0,  0.1,0, 0]

In [6]:
# 姿勢初期化
ri.sendAngleVector(resetAngle,10)

In [7]:
# robotmodel読込み
robot=ri.getRobotModel()
robot.registerEndEffector('arm', ## end-effector
                          EE_LINK, ## tip-link
                          tip_link_to_eef=coordinates(fv(0, 0, 0.05), fv(1/math.sqrt(2), 1/math.sqrt(2), 0,0)),
                          joint_list=robot.jointNames
                         )

In [8]:
# ハンド閉じる
ri.sendAngleVector([0,0,0,0,0,0,0], 1.0,"gripper")

In [9]:
ri.actualAngleVector

array([ 0.00153398,  1.282408  ,  0.79000014,  0.00153398,  0.0966408 ,
       -0.00153398,  0.        ])

In [10]:
import time

class JoyControl(object):
    def __init__(self, scale_pos=0.0015, scale_rot=0.006, robot=None, ri=None):
        self.lock = Lock()
        self.ax = mkshapes.make3DAxis(radius=0.005, length=0.06, axisRatio=0.2)
        self.di = DrawInterface()
        self.di.addObject(self.ax)
        self.sclp = scale_pos
        self.sclr = scale_rot
        self.verbose = False
        self.robot = robot
        self.ri = ri
        self.handopenRad = -PI/3
        self.buff_button = [0,0]
        self.hang = 0
        if self.robot is not None:
            self.ax.newcoords(self.robot.arm.endEffector)
        self.motionMode = 0

        # 角度初期化
        self.angleReset()
        self.robot.setAngleMap({HAND_LINK: 0})
        self.ri.sendAngleVector(self.robot.angleVector(), 1.0,"gripper")

    def callback_msg(self, msg):
        with self.lock:
            #out.append_stdout('out\n')
            if self.verbose:
                out.append_stdout(f'out: {msg}\n')
                rospy.loginfo(f'info: {msg}') ## see /rosout
            pos = fv(msg.axes[0], msg.axes[1], msg.axes[2])
            rpy = fv(-msg.axes[4], msg.axes[3], -msg.axes[5])
            pos *= self.sclp
            rpy *= self.sclr

            if (msg.buttons[0] != 0) and (self.buff_button[0] == 0):
                # 矢印をモデルのEEの座標にリセット
                self.angleReset()
                #out.append_stdout(f'out: {msg}\n')
                
            if (msg.buttons[1] != 0) and (self.buff_button[1] == 0):
                #S_hang = self.robot.angleVector()[7]
                if self.hang == 0:
                    self.robot.setAngleMap({HAND_LINK: self.handopenRad})
                    self.hang = 1
                    self.ri.sendAngleVector(self.robot.angleVector(), 1.0, 'gripper')
                else:
                    self.robot.setAngleMap({HAND_LINK: 0})
                    self.hang = 0
                    self.ri.sendAngleVector(self.robot.angleVector(), 1.0, 'gripper')
                    self.grasp()
                    
            
            if self.motionMode == 0:
                cds = coordinates(pos)
                cds.setRPY(rpy)
            elif self.motionMode == 1:
                cds = coordinates(pos)
            else:
                cds = coordinates()
                cds.setRPY(rpy)
            ##
            self.ax.transform(cds)
            self.IK(self.ax)

            self.buff_button = msg.buttton

    def grasp(self,grasptime = 1.0,maxeffort = 150,graspaxis = 6):
        dt = 0.01
        ef = 0
        for t in range(int(grasptime/dt)):
            ef = self.ri.effortVector[graspaxis]
            if ef > maxeffort:
                self.ri.sendAngleVector(list(self.ri.actualAngleVector), 0.05, 'gripper')
                break
            else:
                pass
            time.sleep(dt)
                
    def IK(self, cds):
        if self.robot is not None:
            self.robot.arm.inverseKinematics(cds)
            vec = self.robot.angleVector()
            self.ri.sendAngleVector(vec,tm=0.016)

    def ax_reset(self):
        self.ax.newcoords(self.robot.arm.endEffector)

    def angleReset(self):
        self.robot.angleVector(resetAngle)
        self.ri.sendAngleVector(robot.angleVector(), 5.0)
        self.ax_reset()
        time.sleep(5)

    def motionChange(self):
        if self.motionMode == 0:
            self.motionMode = 1 # 移動モード１：平行移動
        elif self.motionMode == 1:
            self.motionMode = 2 # 移動モード２：先端の回転のみ
        elif self.motionMode == 2:
            self.motionMode = 0 # 移動モード０：自由移動
        else:
            self.motionMode = 0
    
    def main(self):
        try:
            rospy.get_rostime()
        except:
            rospy.init_node('testjoy', anonymous=False)
                       
        self.sub = rospy.Subscriber('/sp_joy', Joy,
                                    callback=self.callback_msg, queue_size=1)

In [11]:
Pilot = JoyControl(ri = ri,robot = robot)
Pilot.main()